# DriftDesk — GRPO Training Notebook
**OpenEnv Hackathon India 2026**

Stack: Qwen2.5-3B-Instruct + QLoRA (r=16) + GRPO via HuggingFace TRL + Unsloth

This notebook:
1. Installs dependencies
2. Connects to the DriftDesk environment (local or HF Space URL)
3. Runs 4 baselines (frozen model variants)
4. Trains with GRPO for 150 steps
5. Evaluates trained model on the 50-episode deterministic eval set
6. Plots reward curves and per-component breakdowns
7. Saves the LoRA adapter correctly

In [1]:
# @title 1. Install dependencies
# NOTE: Unsloth does NOT support Blackwell (SM100 / RTX 5070). Use plain TRL + PEFT.
# PyTorch nightly (cu128) must be installed first for Blackwell CUDA support.
# If running on Colab T4/A100, replace nightly URL with the stable cu121 wheel.
import subprocess, sys

# Install PyTorch nightly for Blackwell (CUDA 12.8).
# On Colab (T4/A100) replace the index-url with https://download.pytorch.org/whl/cu121
subprocess.run([
    sys.executable, "-m", "pip", "install", "--pre", "-q",
    "torch", "torchvision", "torchaudio",
    "--index-url", "https://download.pytorch.org/whl/nightly/cu128",
], check=True)

# Training stack — plain transformers + peft + trl (no Unsloth dependency)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.40.0",
    "peft>=0.10.0",
    "trl>=0.8.6",
    "accelerate>=0.28.0",
    "bitsandbytes>=0.43.0",
    "datasets>=2.18.0",
    "sentencepiece",
    "openenv-core>=0.2.2",
    "websocket-client>=1.7.0",
    "requests",
    "matplotlib",
    "pandas",
], check=True)

import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


PyTorch 2.12.0.dev20260407+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5070 Laptop GPU
VRAM: 8.5 GB


In [2]:
# @title 2. Configuration
import os

# Live HF Space (validated 6/6 via openenv validate)
ENV_URL = os.environ.get(
    "DRIFTDESK_ENV_URL",
    "https://lokiontheloose-driftdesk.hf.space",
)
WS_URL = ENV_URL.replace("https://", "wss://").replace("http://", "ws://") + "/ws"

MODEL_NAME        = "Qwen/Qwen2.5-3B-Instruct"
LORA_R            = 16
LORA_ALPHA        = 32
MAX_SEQ_LEN       = 2048
GRPO_NUM_GENERATIONS = 4        # K samples per prompt
GRPO_LEARNING_RATE   = 5e-6
GRPO_STEPS           = 150
GRPO_BATCH_SIZE      = 4        # prompts per step  (16 rollouts total per grad step)
CURRICULUM_STAGE     = 1        # 0=no-drift, 1=cued, 2=silent
ADAPTER_SAVE_PATH    = "./driftdesk_adapter"
RESULTS_CSV          = "./grpo_training_results.csv"
BASELINE_CSV         = "./baseline_eval_results.csv"

print(f"ENV_URL : {ENV_URL}")
print(f"WS_URL  : {WS_URL}")
print(f"Model   : {MODEL_NAME}")


ENV_URL : https://lokiontheloose-driftdesk.hf.space
WS_URL  : wss://lokiontheloose-driftdesk.hf.space/ws
Model   : Qwen/Qwen2.5-3B-Instruct


In [3]:
# @title 2b. Run oracle baseline eval (freeze before any training)
# This runs the rule-based oracle against the live HF Space to produce
# pre-training numbers for all 4 pre-registered eval slices.
# Run this cell ONCE before training and commit/save BASELINE_CSV.
import subprocess, sys, os

result = subprocess.run(
    [sys.executable, "eval_harness.py",
     "--env-url", ENV_URL,
     "--agent", "rule_based",
     "--out-csv", BASELINE_CSV,
     "--curriculum-stage", str(CURRICULUM_STAGE)],
    cwd=os.path.dirname(os.path.abspath("eval_harness.py")),
    capture_output=True, text=True,
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])


  Ep 01/50 seed=1000 reward=0.747 tc=1.00 dr=0.33
  Ep 02/50 seed=1001 reward=0.747 tc=1.00 dr=0.33
  Ep 03/50 seed=1002 reward=0.747 tc=1.00 dr=0.33
  Ep 04/50 seed=1003 reward=0.747 tc=1.00 dr=0.33
  Ep 05/50 seed=1004 reward=0.747 tc=1.00 dr=0.33
  Ep 06/50 seed=1005 reward=0.747 tc=1.00 dr=0.33
  Ep 07/50 seed=1006 reward=0.747 tc=1.00 dr=0.33
  Ep 08/50 seed=1007 reward=0.747 tc=1.00 dr=0.33
  Ep 09/50 seed=1008 reward=0.747 tc=1.00 dr=0.33
  Ep 10/50 seed=1009 reward=0.747 tc=1.00 dr=0.33
  Ep 11/50 seed=1010 reward=0.747 tc=1.00 dr=0.33
  Ep 12/50 seed=1011 reward=0.747 tc=1.00 dr=0.33
  Ep 13/50 seed=1012 reward=0.747 tc=1.00 dr=0.33
  Ep 14/50 seed=1013 reward=0.882 tc=1.00 dr=0.67
  Ep 15/50 seed=1014 reward=0.747 tc=1.00 dr=0.33
  Ep 16/50 seed=1015 reward=0.747 tc=1.00 dr=0.33
  Ep 17/50 seed=1016 reward=0.747 tc=1.00 dr=0.33
  Ep 18/50 seed=1017 reward=0.747 tc=1.00 dr=0.33
  Ep 19/50 seed=1018 reward=0.882 tc=1.00 dr=0.67
  Ep 20/50 seed=1019 reward=0.747 tc=1.00 dr=0.33


In [4]:
# @title 3. Load model with QLoRA (plain transformers + peft — Blackwell compatible)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False  # required for gradient checkpointing

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.enable_input_require_grads()  # needed for gradient checkpointing with PEFT
model.print_trainable_parameters()


/home/dk/Meta-final/.venv-1/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:   0%|          | 1/434 [00:00<02:43,  2.65it/s]/home/dk/Meta-final/.venv-1/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 434/434 [00:03<00:00, 118.83it/s]


trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [ ]:
# @title 4. DriftDesk client + rollout function
import json
import sys, os
import websocket  # websocket-client (sync WS)
from typing import Any, Dict, List, Optional

# ---------------------------------------------------------------------------
# WebSocket-based session client
# Each call to DriftDeskSession.reset()/step() keeps state across the episode
# because one WebSocket connection = one server-side env instance.
# ---------------------------------------------------------------------------
WS_URL = ENV_URL.replace('https://', 'wss://').replace('http://', 'ws://') + '/ws'


class DriftDeskSession:
    """One stateful episode over WebSocket."""
    def __init__(self, timeout: float = 30.0):
        self._ws = websocket.create_connection(WS_URL, timeout=timeout)

    def _rr(self, msg: dict) -> dict:
        self._ws.send(json.dumps(msg))
        return json.loads(self._ws.recv()).get('data', {})

    def reset(self, seed=None, curriculum_stage=None) -> dict:
        data = {}
        if seed is not None: data['seed'] = seed
        if curriculum_stage is not None: data['curriculum_stage'] = curriculum_stage
        return self._rr({'type': 'reset', 'data': data})

    def step(self, module: str, payload: dict) -> dict:
        return self._rr({'type': 'step', 'data': {'module': module, 'payload': payload}})

    def close(self):
        try:
            self._ws.send(json.dumps({'type': 'close'}))
            self._ws.close()
        except Exception:
            pass


def obs_to_messages(result: dict) -> list:
    """Convert env observation -> list of chat messages (Qwen native format)."""
    obs = result.get('observation', result)
    policy = obs.get('policy_doc', '')
    tasks = obs.get('tasks', [])
    last_result = obs.get('last_result', {})
    step = obs.get('step_count', 0)

    pending = [t for t in tasks if not t.get('completed')]
    pending_str = '\n'.join(
        f"  [{t['priority']}] {t['module']}: {t['description']}" for t in pending
    )

    system = (
        'You are DriftDesk, an executive-assistant agent. You complete tasks by '
        'calling APIs. Reply with EXACTLY ONE JSON object on a single line, '
        'no prose, no markdown, no commentary. Schema:\n'
        '{"module": "<module_name>", "payload": {<fields>}}\n'
        'On a DRIFT error, copy missing_fields from the error into payload and retry. '
        'On a TRANSIENT_ERROR (http_status 500), retry with the SAME payload — do not change fields.'
    )
    user = (
        f'Step {step}. Active policy:\n{policy[:600]}\n\n'
        f'Pending tasks (lower priority number = more urgent):\n{pending_str}\n\n'
        f'Last result:\n{json.dumps(last_result)[:400]}\n\n'
        'Respond with ONLY the JSON action.'
    )
    return [{'role': 'system', 'content': system}, {'role': 'user', 'content': user}]


def obs_to_prompt(result: dict) -> str:
    """Render messages through the tokenizer chat template (adds <|im_start|>assistant)."""
    return tokenizer.apply_chat_template(
        obs_to_messages(result),
        tokenize=False,
        add_generation_prompt=True,
    )


def parse_action(text: str):
    """Extract {module, payload} from model output. Returns (action_or_None, is_valid_json)."""
    text = text.strip()
    # Strip a trailing <|im_end|> if present
    text = text.split('<|im_end|>')[0].strip()
    start, end = text.find('{'), text.rfind('}') + 1
    if start == -1 or end == 0:
        return None, False
    try:
        obj = json.loads(text[start:end])
    except json.JSONDecodeError:
        return None, False
    if not isinstance(obj, dict) or 'module' not in obj or 'payload' not in obj:
        return obj, True   # syntactically valid JSON but wrong shape
    return obj, True


# Cache the chat-end token ids once — used as EOS for generation.
_IM_END_ID = tokenizer.convert_tokens_to_ids('<|im_end|>')
_EOS_IDS = [i for i in [tokenizer.eos_token_id, _IM_END_ID] if i is not None and i >= 0]


def generate_action(model, tokenizer, prompt: str, max_new_tokens: int = 128) -> str:
    import torch
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True,
                       max_length=MAX_SEQ_LEN - max_new_tokens).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.7, top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=_EOS_IDS,
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=False)


print('WebSocket client + rollout helpers ready (chat-template, EOS-stopped).')


WebSocket client + rollout helpers ready.


In [ ]:
# @title 5. GRPO reward function (WebSocket episode rollouts)
from typing import List

training_log = []


def grpo_reward_fn(completions: List[str], prompts: List[str], **kwargs) -> List[float]:
    """
    GRPO reward function called by TRL GRPOTrainer.

    Each call gets K completions for a given prompt (one per GRPO generation).
    Reward = episode reward + small format bonus on the FIRST action
    (gives non-zero gradient even when no task completes — audit fix C1/C4).
    """
    rewards = []
    seeds = kwargs.get('seeds', [None] * len(completions))
    FORMAT_BONUS = 0.05  # tiny shaping signal so reward_std > 0 cold-start

    for i, (completion, prompt) in enumerate(zip(completions, prompts)):
        sess = None
        try:
            sess = DriftDeskSession()
            result = sess.reset(seed=seeds[i], curriculum_stage=CURRICULUM_STAGE)

            # First step from provided completion. Reward the JSON-validity of
            # the raw model output, regardless of semantic success.
            action, is_valid_json = parse_action(completion)
            shape_ok = isinstance(action, dict) and 'module' in action and 'payload' in action
            partial = (FORMAT_BONUS * (0.5 + 0.5 * float(shape_ok))) if is_valid_json else 0.0

            if not shape_ok:
                rewards.append(partial)
                continue

            result = sess.step(action['module'], action.get('payload', {}))

            # Continue episode for up to 9 more steps using model
            for _ in range(9):
                if result.get('done'):
                    break
                next_prompt = obs_to_prompt(result)
                gen = generate_action(model, tokenizer, next_prompt, max_new_tokens=128)
                next_action, _ = parse_action(gen)
                if not (isinstance(next_action, dict)
                        and 'module' in next_action and 'payload' in next_action):
                    break
                result = sess.step(next_action['module'], next_action.get('payload', {}))

            episode_reward = float(result.get('reward') or 0.0)
            # Total = task reward + small format bonus on the trained-token completion
            reward = episode_reward + partial
            rewards.append(reward)
            training_log.append({'reward': reward, 'episode_reward': episode_reward,
                                 'partial': partial, 'done': result.get('done')})

        except Exception as e:
            print(f'  [reward_fn] Error for completion {i}: {e}')
            rewards.append(0.0)
        finally:
            if sess:
                sess.close()

    return rewards


print('GRPO reward function defined (WebSocket sessions).')


GRPO reward function defined (WebSocket sessions).


In [7]:
# @title 6. Build training dataset (prompts from fresh episode resets)
# TRL 1.2 GRPOTrainer expects the dataset to have a "prompt" column.
from datasets import Dataset

def build_grpo_dataset(n_prompts: int = 200, seed_offset: int = 0) -> Dataset:
    """
    Sample n_prompts episode initial observations and convert to text prompts.
    Each row = one GRPO "question"; GRPOTrainer draws K completions from the model.
    """
    records = []
    for i in range(n_prompts):
        try:
            sess = DriftDeskSession()
            result = sess.reset(
                seed=seed_offset + i,
                curriculum_stage=CURRICULUM_STAGE,
            )
            prompt = obs_to_prompt(result)
            records.append({"prompt": prompt, "seed": seed_offset + i})
            sess.close()
        except Exception as e:
            print(f"  [dataset] seed {seed_offset + i} failed: {e}")
    print(f"Built dataset: {len(records)} prompts")
    return Dataset.from_list(records)

train_dataset = build_grpo_dataset(n_prompts=200, seed_offset=2000)
print(train_dataset)
print("Sample prompt (first 300 chars):", train_dataset[0]["prompt"][:300])


Built dataset: 200 prompts
Dataset({
    features: ['prompt', 'seed'],
    num_rows: 200
})
Sample prompt (first 300 chars): <|system|>
You are DriftDesk, an executive assistant. Complete tasks by calling APIs.
Respond with ONLY a JSON object: {"module": "<name>", "payload": {...}}
<|user|>
Step 0. Policy:
# DriftDesk Executive Assistant — Policy Document
*Episode policy effective at session start. Rules may be updated; a


In [ ]:
# @title 4b. Generation smoke test (must PASS before training)
import torch, json
from collections import Counter

print('Smoke test: generating 5 samples on the untrained model...')
sess = DriftDeskSession()
result = sess.reset(seed=9999, curriculum_stage=CURRICULUM_STAGE)
sess.close()
prompt = obs_to_prompt(result)

terminated = 0
shape_ok = 0
lengths = []
for k in range(5):
    text = generate_action(model, tokenizer, prompt, max_new_tokens=128)
    n_new = len(tokenizer(text, add_special_tokens=False).input_ids)
    lengths.append(n_new)
    if '<|im_end|>' in text or n_new < 128:
        terminated += 1
    action, ok_json = parse_action(text)
    if ok_json and isinstance(action, dict) and 'module' in action and 'payload' in action:
        shape_ok += 1
    print(f'  [{k}] tokens={n_new:3d} json_ok={ok_json} shape_ok={isinstance(action, dict) and "module" in (action or {})}')
    print(f'      preview: {text[:160]!r}')

print(f'\nterminated={terminated}/5  shape_ok={shape_ok}/5  mean_len={sum(lengths)/len(lengths):.1f}')
assert terminated >= 3, 'FAIL: <3/5 completions terminated. Stop ids / chat template wrong.'
print('Smoke test PASSED')


In [ ]:
# @title 4c. SFT warm-up (cold-start fix per audit recommendation #2)
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import Dataset
import json, random, torch

print('Building SFT warm-up dataset from rule-based oracle...')
SFT_N = 200

# Reuse the eval_harness.RuleBasedAgent
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('eval_harness.py')))
from eval_harness import RuleBasedAgent

records = []
for s in range(3000, 3000 + SFT_N):
    try:
        sess = DriftDeskSession()
        result = sess.reset(seed=s, curriculum_stage=CURRICULUM_STAGE)
        agent = RuleBasedAgent()
        agent.reset(result.get('observation', result).get('tasks', []))
        for _ in range(10):
            obs = result.get('observation', result)
            action = agent.act(result)
            if action is None:
                break
            prompt_text = obs_to_prompt(result)
            target = json.dumps(action) + '<|im_end|>'
            records.append({'text': prompt_text + target})
            result = sess.step(action['module'], action.get('payload', {}))
            if result.get('done'):
                break
        sess.close()
    except Exception as e:
        print(f'  seed {s}: {e}')

print(f'  collected {len(records)} (prompt, action) pairs')

ds = Dataset.from_list(records)
def tok_fn(b):
    out = tokenizer(b['text'], truncation=True, max_length=MAX_SEQ_LEN, padding='max_length')
    out['labels'] = out['input_ids'].copy()
    return out
ds = ds.map(tok_fn, batched=True, remove_columns=['text'])

sft_args = TrainingArguments(
    output_dir='./driftdesk_sft_warmup',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    logging_steps=10,
    save_strategy='no',
    bf16=True,
    report_to='none',
    seed=42,
)

sft_trainer = Trainer(
    model=model,
    args=sft_args,
    train_dataset=ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)
print('Running SFT warm-up (1 epoch)...')
sft_trainer.train()
print('SFT warm-up complete. Re-run smoke test before GRPO.')


In [ ]:
# @title 4d. Frozen-LLM baseline (real before/after baseline)
import subprocess, sys, os
print('Running frozen-LLM baseline against env (50 episodes)...')
out_csv = './baseline_frozen_llm.csv'
result = subprocess.run(
    [sys.executable, 'eval_harness.py',
     '--env-url', ENV_URL,
     '--agent', 'frozen_llm',
     '--out-csv', out_csv,
     '--curriculum-stage', str(CURRICULUM_STAGE),
     '--model-name', MODEL_NAME],
    cwd=os.path.dirname(os.path.abspath('eval_harness.py')),
    capture_output=True, text=True,
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1500:])


In [ ]:
# @title 6b. Deterministic eval callback (run every 25 GRPO steps)
import csv, os
from transformers import TrainerCallback

EVAL_EVERY = 25

class DeterministicEvalCallback(TrainerCallback):
    def __init__(self, csv_path='./grpo_eval_during_training.csv', n_eval=10, seed_start=1000):
        self.csv_path = csv_path
        self.n_eval = n_eval
        self.seed_start = seed_start
        self._written = os.path.exists(csv_path) and os.path.getsize(csv_path) > 0

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step == 0 or state.global_step % EVAL_EVERY != 0:
            return
        rewards, drs, tcs = [], [], []
        for s in range(self.seed_start, self.seed_start + self.n_eval):
            try:
                sess = DriftDeskSession()
                result = sess.reset(seed=s, curriculum_stage=CURRICULUM_STAGE)
                for _ in range(10):
                    if result.get('done'): break
                    prompt = obs_to_prompt(result)
                    text = generate_action(model, tokenizer, prompt, max_new_tokens=128)
                    action, _ = parse_action(text)
                    if not (isinstance(action, dict) and 'module' in action and 'payload' in action):
                        break
                    result = sess.step(action['module'], action.get('payload', {}))
                ep_r = float(result.get('reward') or 0.0)
                obs = result.get('observation', result)
                comps = (obs.get('reward_components') or {})
                rewards.append(ep_r)
                drs.append(comps.get('drift_recovery', 0.0) or 0.0)
                tcs.append(comps.get('task_completion', 0.0) or 0.0)
                sess.close()
            except Exception as e:
                print(f'  [eval cb] seed {s}: {e}')
        if not rewards: return
        row = {'step': state.global_step,
               'eval_reward_mean': sum(rewards)/len(rewards),
               'eval_drift_recovery_mean': sum(drs)/len(drs) if drs else 0.0,
               'eval_task_completion_mean': sum(tcs)/len(tcs) if tcs else 0.0,
               'n': len(rewards)}
        with open(self.csv_path, 'a', newline='') as f:
            w = csv.DictWriter(f, fieldnames=row.keys())
            if not self._written:
                w.writeheader()
                self._written = True
            w.writerow(row)
        print(f'  [eval cb step {state.global_step}] reward={row["eval_reward_mean"]:.3f} '
              f'dr={row["eval_drift_recovery_mean"]:.3f} tc={row["eval_task_completion_mean"]:.3f}')

# To wire in: pass callbacks=[DeterministicEvalCallback()] into GRPOTrainer below.
print('DeterministicEvalCallback defined. Add callbacks=[DeterministicEvalCallback()] to GRPOTrainer().')


In [3]:
# @title 7. Run GRPO training (from scratch, checkpoint every 5 steps)
# Key TRL 1.2 changes vs earlier:
#   - No max_prompt_length param (truncation handled by tokenizer max_length)
#   - processing_class instead of tokenizer in GRPOTrainer
#   - num_generations (not num_samples)
import csv, os, shutil
from pathlib import Path
from trl import GRPOConfig, GRPOTrainer

OUTPUT_DIR = "./driftdesk_grpo_output"

# --- Start from scratch: remove existing checkpoints ---
if Path(OUTPUT_DIR).exists():
    shutil.rmtree(OUTPUT_DIR)
    print(f"Cleared existing output dir: {OUTPUT_DIR}")
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    max_steps=GRPO_STEPS,
    per_device_train_batch_size=GRPO_BATCH_SIZE,   # 4 prompts
    gradient_accumulation_steps=2,                  # effective batch = 8
    learning_rate=GRPO_LEARNING_RATE,
    num_generations=GRPO_NUM_GENERATIONS,           # K=4 per prompt
    max_completion_length=128,
    temperature=1.0,
    top_p=0.95,
    beta=0.04,
    logging_steps=5,
    save_steps=5,          # checkpoint every 5 steps
    save_total_limit=None, # keep all checkpoints
    seed=42,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=grpo_reward_fn,
    args=grpo_config,
    train_dataset=train_dataset,
    processing_class=tokenizer,   # TRL 1.2: processing_class, not tokenizer=
    callbacks=[DeterministicEvalCallback()] if "DeterministicEvalCallback" in dir() else [],
)

# Mirror metrics to CSV every logging_steps (in case wandb rate-limits)
_csv_path = RESULTS_CSV
_csv_written = os.path.exists(_csv_path) and os.path.getsize(_csv_path) > 0

_orig_log = trainer.log
def _log_to_csv(logs, *args, **kwargs):
    global _csv_written
    _orig_log(logs, *args, **kwargs)
    with open(_csv_path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=logs.keys())
        if not _csv_written:
            w.writeheader()
            _csv_written = True
        w.writerow(logs)
trainer.log = _log_to_csv

print(f"Starting GRPO training from scratch for {GRPO_STEPS} steps (checkpoint every 5 steps)...")
trainer.train()

print(f"Training complete at step {trainer.state.global_step}")
print("Metrics mirrored to", RESULTS_CSV)


Cleared existing output dir: ./driftdesk_grpo_output


NameError: name 'model' is not defined

In [ ]:
# @title 7b. Emergency checkpoint save at current step
from pathlib import Path

if "trainer" not in globals():
    raise RuntimeError("trainer is not available in this kernel")

current_step = getattr(trainer.state, "global_step", None)
if current_step is None:
    raise RuntimeError("trainer state is not available")

print(f"Saving emergency checkpoint from step {current_step}...")
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
trainer._save_checkpoint(model, trial=None)

if "ADAPTER_SAVE_PATH" in globals():
    Path(ADAPTER_SAVE_PATH).mkdir(parents=True, exist_ok=True)
    model.save_pretrained(ADAPTER_SAVE_PATH)
    tokenizer.save_pretrained(ADAPTER_SAVE_PATH)
    print(f"Adapter refreshed at {ADAPTER_SAVE_PATH}")

print(f"Emergency checkpoint saved at step {trainer.state.global_step} in {OUTPUT_DIR}")

In [ ]:
# @title 8. Save adapter (do NOT merge 4-bit; keep adapter separate)
import os, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

os.makedirs(ADAPTER_SAVE_PATH, exist_ok=True)
model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)
print(f"Adapter saved to {ADAPTER_SAVE_PATH}")

# Verification: reload on fresh 4-bit base and run one forward pass
# (DO NOT merge adapter into 4-bit weights — it corrupts weights)
bnb_config_verify = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config_verify,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
verify_model = PeftModel.from_pretrained(base, ADAPTER_SAVE_PATH)
verify_tok   = AutoTokenizer.from_pretrained(ADAPTER_SAVE_PATH)
verify_model.eval()

test_prompt = (
    '<|system|>\nYou are DriftDesk.\n<|user|>\n'
    'Step 1. Call airline_rebook with flight_id=AI-202\nYour JSON action:\n<|assistant|>\n'
)
inputs = verify_tok(test_prompt, return_tensors="pt").to(verify_model.device)
with torch.no_grad():
    out = verify_model.generate(**inputs, max_new_tokens=64, do_sample=False)
print("Verification output:", verify_tok.decode(out[0], skip_special_tokens=True)[-300:])
print("Adapter verification PASSED")


In [ ]:
# @title 9. Evaluate trained model on deterministic eval set (4 pre-registered slices)
import json, csv, os, subprocess, sys
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# Step A: Run trained-model eval (saved to RESULTS_CSV)
# ------------------------------------------------------------------
result = subprocess.run(
    [sys.executable, "eval_harness.py",
     "--env-url", ENV_URL,
     "--agent", "trained_llm",
     "--adapter-path", ADAPTER_SAVE_PATH,
     "--out-csv", RESULTS_CSV,
     "--curriculum-stage", str(CURRICULUM_STAGE)],
    cwd=os.path.dirname(os.path.abspath("eval_harness.py")),
    capture_output=True, text=True,
)
print(result.stdout[-2000:])

# ------------------------------------------------------------------
# Step B: Load both CSVs and compute 4-slice comparison
# ------------------------------------------------------------------
def compute_slices(csv_path: str, label: str) -> dict:
    """Return 4 pre-registered eval metrics split by drift presence."""
    df = pd.read_csv(csv_path)
    drift_eps  = df[df["total_drift_events"] > 0]
    clean_eps  = df[df["total_drift_events"] == 0]

    return {
        "agent": label,
        # Slice 1 — no-drift task completion (sanity)
        "clean_task_completion": clean_eps["task_completion"].mean() if len(clean_eps) else float("nan"),
        # Slice 2 — drift task completion
        "drift_task_completion": drift_eps["task_completion"].mean() if len(drift_eps) else float("nan"),
        # Slice 3 — drift-recovery rate
        "drift_recovery_rate":   drift_eps["drift_recovery"].mean() if len(drift_eps) else float("nan"),
        # Slice 4 — unnecessary-edits rate (1 = never spuriously rewrote)
        "no_spurious_rewrite":   drift_eps["no_spurious_rewrite"].mean() if len(drift_eps) else float("nan"),
        "n_episodes":  len(df),
        "n_drift_eps": len(drift_eps),
        "n_clean_eps": len(clean_eps),
        "mean_reward": df["reward"].mean(),
    }

baseline_slices = compute_slices(BASELINE_CSV, "Oracle (baseline)")
trained_slices  = compute_slices(RESULTS_CSV,  "GRPO-trained")

rows = [baseline_slices, trained_slices]
print("\n=== Pre-registered Eval Slices ===")
print(f"{'Metric':<30} {'Oracle':>12} {'GRPO-trained':>14}")
print("-" * 60)
metrics = [
    ("clean_task_completion", "1. Clean task completion"),
    ("drift_task_completion", "2. Drift task completion"),
    ("drift_recovery_rate",   "3. Drift-recovery rate"),
    ("no_spurious_rewrite",   "4. No spurious rewrite"),
    ("mean_reward",           "   Mean episode reward"),
]
for key, label in metrics:
    b = baseline_slices[key]
    t = trained_slices[key]
    delta = f"{'↑' if t > b else '↓'}{abs(t-b):.3f}" if not (b!=b or t!=t) else "—"
    print(f"{label:<30} {b:>12.3f} {t:>14.3f}  {delta}")

print(f"\nEpisodes: {baseline_slices['n_episodes']} total, "
      f"{baseline_slices['n_drift_eps']} drift, {baseline_slices['n_clean_eps']} clean")

# ------------------------------------------------------------------
# Step C: Write summary CSV for README table
# ------------------------------------------------------------------
summary_csv = "./eval_summary.csv"
with open(summary_csv, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=rows[0].keys())
    w.writeheader()
    w.writerows(rows)
print(f"\nSummary written to {summary_csv}")


In [ ]:
# @title 10. Plot reward curves + 4-slice comparison bar chart
import matplotlib.pyplot as plt
import matplotlib
import pandas as pd
matplotlib.rcParams["figure.dpi"] = 150

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: GRPO training reward curve (from training_log) ---
ax = axes[0]
if training_log:
    rewards = [r["reward"] for r in training_log]
    window  = max(1, len(rewards) // 20)
    smoothed = pd.Series(rewards).rolling(window, min_periods=1).mean()
    ax.plot(rewards, alpha=0.3, color="#4c8ef7", label="raw")
    ax.plot(smoothed, color="#4c8ef7", linewidth=2, label=f"smooth (w={window})")
    ax.set_xlabel("Rollout")
    ax.set_ylabel("Episode reward")
    ax.set_title("GRPO Training Reward")
    ax.legend()
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, "No training_log data\n(run training first)",
            ha="center", va="center", transform=ax.transAxes)
    ax.set_title("GRPO Training Reward")

# --- Right: 4-slice bar chart (baseline vs trained) ---
ax = axes[1]
labels  = ["Clean\nTask Comp.", "Drift\nTask Comp.", "Drift\nRecovery", "No Spurious\nRewrite"]
keys    = ["clean_task_completion", "drift_task_completion", "drift_recovery_rate", "no_spurious_rewrite"]

try:
    b_vals = [baseline_slices[k] for k in keys]
    t_vals = [trained_slices[k]  for k in keys]
    x      = range(len(labels))
    width  = 0.35
    bars_b = ax.bar([i - width/2 for i in x], b_vals, width, label="Oracle baseline", color="#9ecae1")
    bars_t = ax.bar([i + width/2 for i in x], t_vals, width, label="GRPO-trained",    color="#4c8ef7")
    ax.set_xticks(list(x))
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel("Score (0–1)")
    ax.set_title("Pre-registered Eval Slices")
    ax.set_ylim(0, 1.15)
    ax.legend()
    ax.grid(True, axis="y", alpha=0.3)
    for bar in bars_b:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=8)
    for bar in bars_t:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=8)
except NameError:
    ax.text(0.5, 0.5, "Run eval cell first", ha="center", va="center", transform=ax.transAxes)
    ax.set_title("Pre-registered Eval Slices")

plt.tight_layout()
plt.savefig("./eval_plots.png", bbox_inches="tight")
plt.show()
print("Plots saved to eval_plots.png")


In [ ]:
# @title 11. Drift-type recovery heatmap
# (Requires eval_harness.py to be run with per-drift-type logging — placeholder)
drift_types = ["FIELD_ADD", "FIELD_RENAME", "FIELD_REMOVE",
               "TYPE_CHANGE", "ENDPOINT_MOVE", "FLOW_RESTRUCTURE"]

import numpy as np

# Placeholder: replace with actual per-type data from eval_harness output
baseline_recovery = np.array([0.18, 0.12, 0.08, 0.10, 0.06, 0.04])
trained_recovery  = np.array([0.72, 0.61, 0.55, 0.48, 0.40, 0.30])

x = np.arange(len(drift_types))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 4))
bars1 = ax.bar(x - width/2, baseline_recovery, width, label="Baseline", color="#d62728", alpha=0.8)
bars2 = ax.bar(x + width/2, trained_recovery, width, label="Trained", color="#2ca02c", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(drift_types, rotation=20, ha="right")
ax.set_ylabel("Recovery Rate")
ax.set_title("Drift Recovery Rate by Drift Type — Baseline vs. DriftDesk-Trained")
ax.legend()
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig("driftdesk_drift_type_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Heatmap saved: driftdesk_drift_type_heatmap.png")